# 🖼️ Padding — Notes + Interview
---
> **Simple English** | **Interview Ready**

## 📌 What is Padding? (Simple English)
- Convolution **shrinks** the image (5×5 with 3×3 kernel → 3×3)
- Padding = adding extra **border of zeros** around the image before convolution
- This **prevents shrinking** and preserves spatial information
- Corner pixels are visited fewer times without padding → lose edge info
- Two types: **Valid** (no padding) and **Same** (zero padding)

## 🔑 Types of Padding
| Type | Padding | Output Size | Use When |
|---|---|---|---|
| **Valid** | No padding | Shrinks (n-f+1) | Want smaller output |
| **Same** | Zero padding | Same as input | Want to keep size |

## 🧱 Padding Formula
```
Output size (VALID) = (n - f + 1)          where n=input, f=filter
Output size (SAME)  = n  (always same!)
Padding needed      = (f - 1) / 2  for same output
```
- 3×3 filter → pad=1 | 5×5 filter → pad=2 | 7×7 filter → pad=3

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

# Show effect of padding visually
image = np.random.rand(6,6).astype(np.float32)
img_tf = image.reshape(1,6,6,1)
kernel = np.ones((3,3,1,1), dtype=np.float32) / 9   # blur kernel

out_valid = tf.nn.conv2d(img_tf, kernel, strides=1, padding='VALID').numpy().squeeze()
out_same  = tf.nn.conv2d(img_tf, kernel, strides=1, padding='SAME').numpy().squeeze()

print(f"Input shape   : {image.shape}")
print(f"VALID padding : {out_valid.shape}  (shrinks!)")
print(f"SAME  padding : {out_same.shape}   (stays same!)")

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(image, cmap='Blues');      axes[0].set_title(f'Input {image.shape}')
axes[1].imshow(out_valid, cmap='Blues');  axes[1].set_title(f'VALID → {out_valid.shape} (shrinks)')
axes[2].imshow(out_same, cmap='Blues');   axes[2].set_title(f'SAME  → {out_same.shape} (preserved)')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# Zero padding visualization
def add_padding(image, pad):
    return np.pad(image, pad_width=pad, mode='constant', constant_values=0)

img_5x5 = np.ones((5,5)) * 128  # gray image
img_padded = add_padding(img_5x5, pad=1)

fig, axes = plt.subplots(1, 2, figsize=(8,4))
axes[0].imshow(img_5x5, cmap='gray', vmin=0, vmax=255)
axes[0].set_title(f'Original: {img_5x5.shape}')
axes[1].imshow(img_padded, cmap='gray', vmin=0, vmax=255)
axes[1].set_title(f'After Padding (p=1): {img_padded.shape}')
# Highlight padding border
for ax, data in zip(axes, [img_5x5, img_padded]):
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            ax.text(j, i, int(data[i,j]), ha='center', va='center', fontsize=9)
plt.tight_layout(); plt.show()
print("Zeros added as border → filter has same-sized neighborhood at edges too!")

In [ ]:
# In Keras: padding='valid' vs padding='same'
import tensorflow as tf

model_valid = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3,3), padding='valid', input_shape=(28,28,1)),
])
model_same = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3,3), padding='same', input_shape=(28,28,1)),
])
model_valid.build(); model_same.build()
print("Input shape     : (28, 28, 1)")
print(f"padding='valid' → {model_valid.layers[0].output_shape}")
print(f"padding='same'  → {model_same.layers[0].output_shape}")
print("\npadding='same' is commonly used to keep spatial dimensions!")

## 🗣️ Interview Q&A

**Q: Why do we need padding?**
> Two reasons: (1) Prevents output from shrinking with each layer — deep networks would lose too much info, (2) Ensures edge/corner pixels are processed equally as center pixels.

**Q: What is 'same' padding?**
> Adds zeros around the border so output has the same height/width as input. Amount of padding = (kernel_size - 1) / 2. For 3×3 kernel → pad=1.

**Q: What is 'valid' padding?**
> No padding added. Output shrinks. Formula: output = (input - kernel + 1). Use when you want downsampling.

**Q: What value is used for padding?**
> Usually **zeros** (zero-padding). Occasionally reflection padding or replication padding is used in advanced architectures.